# Notebook 05 — CoreML Export & Neural Engine Benchmark

Take the fine-tuned ViT from notebook 03 and ship it to an iPhone.

Pipeline: PyTorch checkpoint → TorchScript trace → coremltools → `.mlpackage`

---

## Table of Contents

1. [Setup & Install](#part-1)
2. [Load & Verify PyTorch Model](#part-2)
3. [Export to CoreML](#part-3)
4. [Verify Output Parity](#part-4)
5. [Latency Benchmark](#part-5)
6. [Model Metadata & Save](#part-6)

---

<a id="part-1"></a>
## Part 1 — Setup & Install

In [1]:
# Install coremltools if not present
try:
    import coremltools as ct
    print(f'coremltools {ct.__version__} already installed')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'coremltools'], check=True)
    import coremltools as ct
    print(f'coremltools {ct.__version__} installed')

Torch version 2.10.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


coremltools 9.0 already installed


In [2]:
import torch
import timm
import numpy as np
import time
import json
import coremltools as ct
from pathlib import Path
from torchvision import transforms
from PIL import Image

CLASSES = ['beach', 'forest', 'mountain', 'kitchen', 'bedroom',
           'street', 'restaurant', 'office', 'living room', 'park']

CHECKPOINT   = Path('../models/vit_finetuned_sun397.pth')
MLPACKAGE    = Path('../models/vit_scene_classifier.mlpackage')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'coremltools: {ct.__version__}')
print(f'torch:       {torch.__version__}')
print(f'MPS available: {torch.backends.mps.is_available()}')

coremltools: 9.0
torch:       2.10.0
MPS available: True


[↑ Back to top](#part-1)

---

<a id="part-2"></a>
## Part 2 — Load & Verify PyTorch Model

Load the fine-tuned checkpoint from notebook 03 and run a quick sanity check on CPU.  
We use CPU here because TorchScript tracing must happen on CPU for CoreML export.

In [3]:
# Load on CPU — tracing for CoreML must be done on CPU
model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=10)

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT}\n'
        'Run notebook 03 first to train and save the fine-tuned model.'
    )

state = torch.load(CHECKPOINT, map_location='cpu')
model.load_state_dict(state)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Loaded: {CHECKPOINT}')
print(f'Parameters: {total_params / 1e6:.1f}M')

Loaded: ../models/vit_finetuned_sun397.pth
Parameters: 85.8M


In [4]:
# Normalisation used during training — must match exactly for export
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

# Quick forward pass — verify shape and output
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    logits = model(dummy)
probs = torch.softmax(logits, dim=-1)
print(f'Output shape:  {logits.shape}')     # (1, 10)
print(f'Probs sum:     {probs.sum().item():.6f}')  # should be ~1.0
print(f'Top class:     {CLASSES[probs.argmax().item()]} ({probs.max().item():.1%})')

Output shape:  torch.Size([1, 10])
Probs sum:     1.000000
Top class:     forest (94.3%)


[↑ Back to top](#part-1)

---

<a id="part-3"></a>
## Part 3 — Export to CoreML

**Two-step process:**

1. **TorchScript trace** — runs a forward pass with example input to record all tensor operations into a static graph. This is what coremltools reads.
2. **coremltools convert** — translates the TorchScript graph to CoreML's `.mlpackage` format, which the Neural Engine can execute.

**`compute_units=ALL`** tells CoreML to use Neural Engine + GPU + CPU and auto-route each op to the fastest available hardware.

In [5]:
# Step 1: TorchScript trace
# Must be on CPU with a representative input
example_input = torch.randn(1, 3, 224, 224)

print('Tracing model...')
with torch.no_grad():
    traced = torch.jit.trace(model, example_input)
print('Trace complete.')

# Verify traced model produces same output as original
with torch.no_grad():
    out_original = model(example_input)
    out_traced   = traced(example_input)

max_diff = (out_original - out_traced).abs().max().item()
print(f'Max output diff (original vs traced): {max_diff:.2e}')  # should be ~1e-6 or smaller

Tracing model...


/Users/preetichauhan/anaconda3/envs/codingenv/lib/python3.11/site-packages/torch/__init__.py:2228: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Trace complete.
Max output diff (original vs traced): 0.00e+00


In [6]:
# Step 2: Convert to CoreML
# Input: normalized float tensor (1, 3, 224, 224)
# Output: raw logits (1, 10) — we add softmax inside Swift, or use classifierConfig

print('Converting to CoreML (this takes ~1–2 min)...')

mlmodel = ct.convert(
    traced,
    inputs=[
        ct.TensorType(
            name='image',
            shape=(1, 3, 224, 224),
            dtype=np.float32
        )
    ],
    outputs=[
        ct.TensorType(name='logits', dtype=np.float32)
    ],
    compute_units=ct.ComputeUnit.ALL,
    minimum_deployment_target=ct.target.iOS16,
    convert_to='mlprogram',      # .mlpackage format (vs legacy .mlmodel)
)

print('Conversion complete.')
print(mlmodel)

Converting to CoreML (this takes ~1–2 min)...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 284.95 passes/s]


Conversion complete.
input {
  name: "image"
  type {
    multiArrayType {
      shape: 1
      shape: 3
      shape: 224
      shape: 224
      dataType: FLOAT32
    }
  }
}
output {
  name: "logits"
  type {
    multiArrayType {
      shape: 1
      shape: 10
      dataType: FLOAT32
    }
  }
}
metadata {
  userDefined {
    key: "com.github.apple.coremltools.version"
    value: "9.0"
  }
  userDefined {
    key: "com.github.apple.coremltools.source"
    value: "torch==2.10.0"
  }
  userDefined {
    key: "com.github.apple.coremltools.source_dialect"
    value: "TorchScript"
  }
  userDefined {
    key: "com.github.apple.coremltools.conversion_date"
    value: "2026-03-13"
  }
}



[↑ Back to top](#part-1)

---

<a id="part-4"></a>
## Part 4 — Verify Output Parity

Run the same image through PyTorch and CoreML. The predicted class should match. Raw logit values may differ slightly due to float32 rounding in the conversion, but top-1 class must be identical.

In [7]:
# Load a sample image from SUN397 for parity check
from datasets import load_dataset
hf_test = load_dataset('pc-ml-dl/sun397', split='test')
TARGET_INDICES = [46, 159, 244, 213, 48, 335, 300, 255, 227, 265]
for row in hf_test:
    if row['label'] in TARGET_INDICES:
        pil_img = row['image'].convert('RGB')
        true_label = TARGET_INDICES.index(row['label'])
        print(f'Sample image class: {CLASSES[true_label]}')
        break

# Preprocess for PyTorch and CoreML
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(pil_img).unsqueeze(0)  # (1,3,224,224) float32
input_np     = input_tensor.numpy()              # CoreML expects numpy
print(f'input_tensor shape: {input_tensor.shape}, dtype: {input_tensor.dtype}')

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Sample image class: forest
input_tensor shape: torch.Size([1, 3, 224, 224]), dtype: torch.float32


In [8]:
# PyTorch prediction
with torch.no_grad():
    pt_logits = model(input_tensor)               # (1, 10)
    pt_probs  = torch.softmax(pt_logits, dim=-1)
    pt_class  = pt_probs.argmax().item()

# CoreML prediction
cml_out    = mlmodel.predict({'image': input_np})
cml_logits = cml_out['logits']                    # numpy (1, 10) or (10,)
cml_logits = np.array(cml_logits).flatten()
cml_probs  = np.exp(cml_logits) / np.exp(cml_logits).sum()
cml_class  = int(np.argmax(cml_logits))

print(f'PyTorch  → {CLASSES[pt_class]:15s} ({pt_probs[0, pt_class].item():.1%})')
print(f'CoreML   → {CLASSES[cml_class]:15s} ({cml_probs[cml_class]:.1%})')
print(f'Match: {"✓" if pt_class == cml_class else "✗"}')

# Logit-level diff
pt_logits_np = pt_logits.squeeze().numpy()
max_logit_diff = np.abs(pt_logits_np - cml_logits).max()
print(f'Max logit diff: {max_logit_diff:.4f}')

PyTorch  → forest          (99.9%)
CoreML   → forest          (99.9%)
Match: ✓
Max logit diff: 0.0082


In [9]:
# Assertions
assert pt_class == cml_class, \
    f'Top-1 class mismatch: PyTorch={CLASSES[pt_class]}, CoreML={CLASSES[cml_class]}'
assert max_logit_diff < 1.0, \
    f'Logit diff too large: {max_logit_diff:.4f} — conversion may have failed'

print('All parity assertions passed.')

All parity assertions passed.


[↑ Back to top](#part-1)

---

<a id="part-5"></a>
## Part 5 — Latency Benchmark

Compare inference latency across three compute configurations:
- **ALL** — Neural Engine + GPU + CPU (CoreML auto-routes)
- **CPU_AND_NE** — Neural Engine + CPU only
- **CPU_ONLY** — baseline, no accelerator

The Neural Engine runs int8/float16 ops in parallel at extremely low wattage — that's why on-device ML is power-efficient on Apple Silicon.

In [10]:
def benchmark_coreml(model_path, input_np, n_warmup=10, n_runs=50):
    """
    Benchmark a saved .mlpackage.
    Returns mean and p95 latency in milliseconds.
    """
    import coremltools as ct
    m = ct.models.MLModel(str(model_path))

    # Warmup — first few runs are slower (model loading, JIT)
    for _ in range(n_warmup):
        m.predict({'image': input_np})

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        m.predict({'image': input_np})
        times.append((time.perf_counter() - t0) * 1000)

    return {
        'mean_ms':  np.mean(times),
        'p50_ms':   np.percentile(times, 50),
        'p95_ms':   np.percentile(times, 95),
        'min_ms':   np.min(times),
    }

print('benchmark_coreml defined.')

benchmark_coreml defined.


In [11]:
# Convert and save models for each compute unit setting
compute_configs = {
    'ALL':        ct.ComputeUnit.ALL,
    'CPU_AND_NE': ct.ComputeUnit.CPU_AND_NE,
    'CPU_ONLY':   ct.ComputeUnit.CPU_ONLY,
}

saved_paths = {}
for name, cu in compute_configs.items():
    path = MODELS_DIR / f'vit_scene_{name.lower()}.mlpackage'
    print(f'Converting {name}...')
    m = ct.convert(
        traced,
        inputs=[ct.TensorType(name='image', shape=(1, 3, 224, 224), dtype=np.float32)],
        outputs=[ct.TensorType(name='logits', dtype=np.float32)],
        compute_units=cu,
        minimum_deployment_target=ct.target.iOS16,
        convert_to='mlprogram',
    )
    m.save(str(path))
    saved_paths[name] = path
    print(f'  Saved → {path.name}')

print('All variants saved.')

Converting ALL...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 304.07 passes/s]


  Saved → vit_scene_all.mlpackage
Converting CPU_AND_NE...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 313.57 passes/s]


  Saved → vit_scene_cpu_and_ne.mlpackage
Converting CPU_ONLY...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 299.63 passes/s]


  Saved → vit_scene_cpu_only.mlpackage
All variants saved.


In [12]:
# Run benchmark
results = {}
for name, path in saved_paths.items():
    print(f'Benchmarking {name}...')
    results[name] = benchmark_coreml(path, input_np, n_warmup=10, n_runs=50)

print('\n=== Latency Results ===')
print(f'{"Config":15s} {"Mean":>10s} {"P50":>10s} {"P95":>10s} {"Min":>10s}')
print('-' * 55)
for name, r in results.items():
    print(f"{name:15s} {r['mean_ms']:9.1f}ms {r['p50_ms']:9.1f}ms "
          f"{r['p95_ms']:9.1f}ms {r['min_ms']:9.1f}ms")

# Speedup vs CPU_ONLY
if 'CPU_ONLY' in results and 'ALL' in results:
    speedup = results['CPU_ONLY']['mean_ms'] / results['ALL']['mean_ms']
    print(f'\nNeural Engine speedup (ALL vs CPU_ONLY): {speedup:.1f}x')

Benchmarking ALL...
Benchmarking CPU_AND_NE...
Benchmarking CPU_ONLY...

=== Latency Results ===
Config                Mean        P50        P95        Min
-------------------------------------------------------
ALL                   8.9ms       8.9ms       9.1ms       8.7ms
CPU_AND_NE            9.5ms       9.2ms      11.0ms       8.7ms
CPU_ONLY              9.1ms       8.9ms      10.5ms       8.0ms

Neural Engine speedup (ALL vs CPU_ONLY): 1.0x


In [13]:
# Also benchmark PyTorch on MPS for comparison
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model_mps = model.to(device)
input_mps = input_tensor.to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model_mps(input_mps)

times_pt = []
for _ in range(50):
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model_mps(input_mps)
    times_pt.append((time.perf_counter() - t0) * 1000)

print(f'PyTorch MPS:  mean={np.mean(times_pt):.1f}ms  p50={np.percentile(times_pt,50):.1f}ms')
print('(MPS does not use Neural Engine — CoreML ALL should be faster)')

PyTorch MPS:  mean=8.8ms  p50=8.7ms
(MPS does not use Neural Engine — CoreML ALL should be faster)


[↑ Back to top](#part-1)

---

<a id="part-6"></a>
## Part 6 — Model Metadata & Final Save

Add metadata to the `.mlpackage`: description, class labels, author, version.  
This is what Xcode shows in the model inspector and what Swift uses to get class name strings directly from the model.

In [14]:
# Reload the ALL-compute-units model and add metadata
final_model = ct.models.MLModel(str(saved_paths['ALL']))

# Save class labels and metadata as user-defined metadata (Swift-readable)
final_model.user_defined_metadata['classes']     = json.dumps(CLASSES)
final_model.user_defined_metadata['num_classes'] = str(len(CLASSES))
final_model.user_defined_metadata['image_mean']  = '0.485,0.456,0.406'
final_model.user_defined_metadata['image_std']   = '0.229,0.224,0.225'
final_model.user_defined_metadata['author']      = 'Smart Scene Classifier'
final_model.user_defined_metadata['version']     = '1.0'
final_model.user_defined_metadata['description'] = (
    'ViT-B/16 fine-tuned on SUN397 (10 scene classes). '
    'Input: normalized (1,3,224,224) float32. '
    'Output: logits (1,10) — apply softmax for probabilities.'
)

# Save as the canonical shipped model
final_model.save(str(MLPACKAGE))
print(f'Saved → {MLPACKAGE}')
print(f'Size:    {sum(f.stat().st_size for f in MLPACKAGE.rglob("*") if f.is_file()) / 1e6:.1f} MB')

Saved → ../models/vit_scene_classifier.mlpackage
Size:    171.7 MB


In [15]:
# Verify metadata round-trips correctly
loaded = ct.models.MLModel(str(MLPACKAGE))
meta   = loaded.user_defined_metadata

recovered_classes = json.loads(meta['classes'])
assert recovered_classes == CLASSES, 'Class label round-trip failed'
assert int(meta['num_classes']) == 10, 'num_classes mismatch'

# Final parity check on the saved model
final_out    = loaded.predict({'image': input_np})
final_logits = np.array(final_out['logits']).flatten()
final_class  = int(np.argmax(final_logits))
assert final_class == pt_class, 'Final model class mismatch vs PyTorch'

print(f'Classes:  {recovered_classes}')
print(f'Pred:     {CLASSES[final_class]}')
print('All metadata and parity assertions passed.')

Classes:  ['beach', 'forest', 'mountain', 'kitchen', 'bedroom', 'street', 'restaurant', 'office', 'living room', 'park']
Pred:     forest
All metadata and parity assertions passed.


In [16]:
# Summary
print('\n=== Export Summary ===')
print(f'Model:          vit_base_patch16_224 (fine-tuned on SUN397 10-class subset)')
print(f'Input:          (1, 3, 224, 224) float32, ImageNet-normalized')
print(f'Output:         logits (1, 10) — apply softmax for class probabilities')
print(f'Classes:        {CLASSES}')
print(f'Saved to:       {MLPACKAGE}')
print(f'Compute units:  ALL (Neural Engine + GPU + CPU)')
print(f'Min iOS target: iOS 16')
print(f'Format:         mlprogram (.mlpackage)')
if 'ALL' in results:
    print(f'Latency (NE):   {results["ALL"]["mean_ms"]:.1f}ms mean')
print()
print('Swift usage:')
print('  let model = try vit_scene_classifier(configuration: MLModelConfiguration())')
print('  let input = vit_scene_classifierInput(image: pixelBuffer)')
print('  let output = try model.prediction(input: input)')


=== Export Summary ===
Model:          vit_base_patch16_224 (fine-tuned on SUN397 10-class subset)
Input:          (1, 3, 224, 224) float32, ImageNet-normalized
Output:         logits (1, 10) — apply softmax for class probabilities
Classes:        ['beach', 'forest', 'mountain', 'kitchen', 'bedroom', 'street', 'restaurant', 'office', 'living room', 'park']
Saved to:       ../models/vit_scene_classifier.mlpackage
Compute units:  ALL (Neural Engine + GPU + CPU)
Min iOS target: iOS 16
Format:         mlprogram (.mlpackage)
Latency (NE):   8.9ms mean

Swift usage:
  let model = try vit_scene_classifier(configuration: MLModelConfiguration())
  let input = vit_scene_classifierInput(image: pixelBuffer)
  let output = try model.prediction(input: input)


[↑ Back to top](#part-1)

---

**App-01 complete.** The fine-tuned ViT is now a `.mlpackage` ready to drop into Xcode.

**Next →** `app-02` — Live Object Detector (DETR / YOLOv8)